In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import re
import scipy
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [3]:
 from google.colab import drive
 drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/MyDrive/data_goodreads.csv")
df = df.head(300000)

In [5]:
def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return None

def parse_genres(x):
    if not isinstance(x, str):
        return []
    return re.findall(r'"(.*?)"', x)

def encode_genres(genres):
  vec = np.zeros(30, dtype=np.float32)
  for g in genres:
      if g in genre_to_idx:
          vec[genre_to_idx[g]] = 1.0
  return vec

bad_rows = df[df["genres"].apply(lambda x: safe_eval(x) is None)]
df["genres"] = df["genres"].apply(parse_genres)
top30 = df["genres"].explode().value_counts().head(30)
top30 = df["genres"].explode().value_counts().head(30).index.tolist()
genre_to_idx = {g: i for i, g in enumerate(top30)}
y = np.stack(df["genres"].apply(encode_genres))

In [28]:
class MLP_basic(nn.Module):
    def __init__(self, input_dim=5000, num_classes=30):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, num_classes)
          )

    def forward(self, x):
        return self.net(x)

class MLP_w2v(nn.Module):
    def __init__(self, input_dim=300, num_classes=30):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, num_classes)
          )

    def forward(self, x):
        return self.net(x)

In [21]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y.float())

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)

    return total_loss / total_samples

@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        probs = torch.sigmoid(logits)
        preds = (probs > threshold).cpu().numpy()

        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())

        total_loss += loss.item() * x.size(0)

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    micro_f1 = f1_score(all_targets, all_preds, average="micro")

    return total_loss / len(loader.dataset), micro_f1

In [8]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class SparseTextDataset(Dataset):
    def __init__(self, X_sparse, y):
        self.X = X_sparse
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x_row = self.X[idx].toarray().flatten()
        return torch.tensor(x_row, dtype=torch.float32), self.y[idx]

In [31]:
def load_X(embedding_method= 'w2v'):
  if embedding_method == 'w2v':
    container = np.load('/content/drive/MyDrive/data/w2v_emb.npz')
    X = container['w2v_vectors']
    X = X[:300000]

  if embedding_method == 'tfidf':
    X = scipy.sparse.load_npz('/content/drive/MyDrive/data/tfidf_emb.npz')
    X = X[:300000]

  if embedding_method == 'bow':
    X = scipy.sparse.load_npz('/content/drive/MyDrive/data/bow_emb.npz')
    X = X[:300000]

  return X




def create_loader(X, y, embedding_method= 'w2v'):

  if embedding_method == 'w2v':
    dataset = TextDataset(X, y)

  else:
    dataset = SparseTextDataset(X, y)

  loader = DataLoader(dataset, batch_size=32, shuffle=True)

  return loader

def experiment(X, y, embedding_method = 'w2v'):

  device = "cuda" if torch.cuda.is_available() else "cpu"

  if embedding_method == 'w2v':
    model = MLP_w2v().to(device)

  else:
    model = MLP_basic().to(device)

  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

  pos_counts = y.sum(axis=0)
  neg_counts = len(y) - pos_counts

  pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(device)

  criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

  #criterion = nn.BCEWithLogitsLoss()

  X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
  )

  train_loader = create_loader(
      X_train,
      y_train,
      embedding_method= embedding_method
  )

  test_loader = create_loader(
      X_test,
      y_test,
      embedding_method= embedding_method
  )

  for epoch in range(10):
      train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
      val_loss, val_f1 = evaluate(model, test_loader, criterion, device)

      print(f"Epoch {epoch}")
      print(f"Train loss: {train_loss:.4f}")
      print(f"Val loss:   {val_loss:.4f}")
      print(f"Val micro-F1: {val_f1:.4f}")

In [32]:
# w2v
m = 'w2v'
X = load_X(embedding_method= m)
experiment(X, y, embedding_method= m)

Epoch 0
Train loss: 0.7928
Val loss:   0.7272
Val micro-F1: 0.3295
Epoch 1
Train loss: 0.7038
Val loss:   0.6994
Val micro-F1: 0.3474
Epoch 2
Train loss: 0.6770
Val loss:   0.6883
Val micro-F1: 0.3497
Epoch 3
Train loss: 0.6599
Val loss:   0.6789
Val micro-F1: 0.3519
Epoch 4
Train loss: 0.6456
Val loss:   0.6733
Val micro-F1: 0.3620
Epoch 5
Train loss: 0.6333
Val loss:   0.6776
Val micro-F1: 0.3599
Epoch 6
Train loss: 0.6220
Val loss:   0.6689
Val micro-F1: 0.3558
Epoch 7
Train loss: 0.6117
Val loss:   0.6794
Val micro-F1: 0.3703
Epoch 8
Train loss: 0.6022
Val loss:   0.6770
Val micro-F1: 0.3622
Epoch 9
Train loss: 0.5929
Val loss:   0.6935
Val micro-F1: 0.3600


In [19]:
# bow
m = 'tfidf'
X = load_X(embedding_method=m)
experiment(X, y, embedding_method=m)

Epoch 0
Train loss: 0.7105
Val loss:   0.6529
Val micro-F1: 0.2368
Epoch 1
Train loss: 0.5588
Val loss:   0.6590
Val micro-F1: 0.2771
Epoch 2
Train loss: 0.4249
Val loss:   0.7914
Val micro-F1: 0.3228


In [20]:
# bow
m = 'bow'
X = load_X(embedding_method=m)
experiment(X, y, embedding_method=m)

Epoch 0
Train loss: 0.7182
Val loss:   0.6695
Val micro-F1: 0.2445
Epoch 1
Train loss: 0.5362
Val loss:   0.7089
Val micro-F1: 0.2776
Epoch 2
Train loss: 0.3989
Val loss:   0.9151
Val micro-F1: 0.3194


Dla 3 epoch, bow i tfidf wykonywaly sie 2 razy dłużej od w2v, a sieć nadal jest bardzo mała. Co będzie później???